In [9]:
# add imports here
from fastapi import FastAPI, Request, Response
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel
from typing import Optional
from fastapi.responses import JSONResponse

In [10]:
def clear_route(app: FastAPI, path: str, method: str):
    method = method.upper()
    app.router.routes = [
        r for r in app.router.routes
        if not (getattr(r, "path", None) == path and method in getattr(r, "methods", set()))
    ]

print("Imports OK - FastAPI app can now be built task by task.")

Imports OK - FastAPI app can now be built task by task.


## Task 0

In [12]:
app = FastAPI(title="Task API", version="1.0")

@app.get("/")
def hello():
    return {"message": "Hello, server!"}

# --- checkpoint: curl -i http://localhost:8000/  ---
client = TestClient(app)
r = client.get("/")
print("Status:", r.status_code)
print("Body  :", r.json())
assert r.status_code == 200
print("\nTask 0 checkpoint completed.")


Status: 200
Body  : {'message': 'Hello, server!'}

Task 0 checkpoint completed.


## Task 1

In [14]:
clear_route(app, "/", "GET")

@app.get("/")
def root():
    
    # now the real API description the brief asks for.
    return {"name": "Task API", "version": "1.0", "endpoints": ["/tasks"]}

@app.get("/health")
def health():
    return {"status": "ok"}

client = TestClient(app)

r1 = client.get("/")
print("GET /        -", r1.status_code, r1.json())
assert r1.status_code == 200

r2 = client.get("/health")
print("GET /health  -", r2.status_code, r2.json())
assert r2.status_code == 200

print("\nTask 1 checkpoint completed.")


GET /        - 200 {'name': 'Task API', 'version': '1.0', 'endpoints': ['/tasks']}
GET /health  - 200 {'status': 'ok'}

Task 1 checkpoint completed.


## Task 2

In [16]:
tasks = [
    {"id": 1, "title": "Buy milk", "done": False},
    {"id": 2, "title": "Write weekly report", "done": True},
    {"id": 3, "title": "Walk the dog", "done": False},
]
next_id = 4  # next free id to hand out

def error(status_code: int, message: str) -> JSONResponse:
    
    # Every error in this API is JSON: {"error": "..."} with the right status code.
    return JSONResponse(status_code=status_code, content={"error": message})


def find_task(task_id: int):
    return next((t for t in tasks if t["id"] == task_id), None)

@app.get("/tasks")
def list_tasks():
    return tasks

@app.get("/tasks/{task_id}")
def get_task(task_id: int):
    task = find_task(task_id)
    if task is None:
        return error(404, f"Task {task_id} not found")
    return task

client = TestClient(app)


r1 = client.get("/tasks/1")
print("GET /tasks/1  -", r1.status_code, r1.json())
assert r1.status_code == 200



r2 = client.get("/tasks/99")
print("GET /tasks/99 -", r2.status_code, r2.json())
assert r2.status_code == 404


print("\nTask 2 checkpoint completed.")


GET /tasks/1  - 200 {'id': 1, 'title': 'Buy milk', 'done': False}
GET /tasks/99 - 404 {'error': 'Task 99 not found'}

Task 2 checkpoint completed.


## Task 3

In [17]:
class TaskCreate(BaseModel):
    title: Optional[str] = None


@app.post("/tasks", status_code=201)
def create_task(payload: TaskCreate, request: Request, response: JSONResponse = None):
    global next_id
    if not payload.title or not payload.title.strip():
        return error(400, "title is required and cannot be empty")
    new_task = {"id": next_id, "title": payload.title.strip(), "done": False}
    tasks.append(new_task)
    next_id += 1
    return JSONResponse(status_code=201, content=new_task)

client = TestClient(app)


# curl -i -X POST http://localhost:8000/tasks -H "Content-Type: application/json" -d '{"title":"Buy milk"}'
r1 = client.post("/tasks", json={"title": "Read a book"})
print("POST /tasks {title:'Read a book'}  -", r1.status_code, r1.json())
assert r1.status_code == 201


r2 = client.get("/tasks")
print("GET /tasks now has", len(r2.json()), "tasks")
assert len(r2.json()) == 4


# Posting {} returns 400
r3 = client.post("/tasks", json={})
print("POST /tasks {}                     -", r3.status_code, r3.json())
assert r3.status_code == 400

print("\nTask 3 checkpoint completed.")

POST /tasks {title:'Read a book'}  - 201 {'id': 4, 'title': 'Read a book', 'done': False}
GET /tasks now has 4 tasks
POST /tasks {}                     - 400 {'error': 'title is required and cannot be empty'}

Task 3 checkpoint completed.


## Task 4

In [18]:
class TaskUpdate(BaseModel):
    title: Optional[str] = None
    done: Optional[bool] = None


@app.put("/tasks/{task_id}")
def update_task(task_id: int, payload: TaskUpdate):
    task = find_task(task_id)
    if task is None:
        return error(404, f"Task {task_id} not found")

    if payload.title is None and payload.done is None:
        return error(400, "request body must include at least 'title' or 'done'")
    if payload.title is not None and not payload.title.strip():
        return error(400, "title cannot be empty")

    if payload.title is not None:
        task["title"] = payload.title.strip()
    if payload.done is not None:
        task["done"] = payload.done
    return task

@app.delete("/tasks/{task_id}", status_code=204)
def delete_task(task_id: int):
    task = find_task(task_id)
    if task is None:
        return error(404, f"Task {task_id} not found")
    tasks.remove(task)
    return Response(status_code=204)  # true empty body, per "204 No Content"

client = TestClient(app)


r1 = client.put("/tasks/1", json={"done": True})
print("PUT /tasks/1 {done:true} -", r1.status_code, r1.json())
assert r1.status_code == 200 and r1.json()["done"] is True


r2 = client.put("/tasks/99", json={"done": True})
print("PUT /tasks/99            -", r2.status_code, r2.json())
assert r2.status_code == 404


r3 = client.put("/tasks/1", json={})
print("PUT /tasks/1 {}          -", r3.status_code, r3.json())
assert r3.status_code == 400


r4 = client.delete("/tasks/1")
print("DELETE /tasks/1          -", r4.status_code, "(no body)" if not r4.content else r4.content)
assert r4.status_code == 204


r5 = client.delete("/tasks/1")
print("DELETE /tasks/1 (again)  -", r5.status_code, r5.json())
assert r5.status_code == 404


print("\nTask 4 checkpoint completed.")


PUT /tasks/1 {done:true} - 200 {'id': 1, 'title': 'Buy milk', 'done': True}
PUT /tasks/99            - 404 {'error': 'Task 99 not found'}
PUT /tasks/1 {}          - 400 {'error': "request body must include at least 'title' or 'done'"}
DELETE /tasks/1          - 204 (no body)
DELETE /tasks/1 (again)  - 404 {'error': 'Task 1 not found'}

Task 4 checkpoint completed.


### Task 4 (the full CRUD cycle)

In [19]:
client = TestClient(app)

before = len(client.get("/tasks").json())

c = client.post("/tasks", json={"title": "Full-cycle demo task"})
print("1) CREATE -", c.status_code, c.json())
assert c.status_code == 201
tid = c.json()["id"]


u = client.put(f"/tasks/{tid}", json={"title": "Full-cycle demo task (edited)"})
print("2) UPDATE -", u.status_code, u.json())
assert u.status_code == 200


d_done = client.put(f"/tasks/{tid}", json={"done": True})
print("3) MARK DONE -", d_done.status_code, d_done.json())
assert d_done.status_code == 200 and d_done.json()["done"] is True


d = client.delete(f"/tasks/{tid}")
print("4) DELETE -", d.status_code)
assert d.status_code == 204


after = client.get("/tasks").json()
print("5) GET /tasks - back to", len(after), "tasks")
assert len(after) == before

print("\nFull CRUD lifecycle checkpoint completed - 201, 200, 200, 204, 200 in sequence.")


1) CREATE - 201 {'id': 5, 'title': 'Full-cycle demo task', 'done': False}
2) UPDATE - 200 {'id': 5, 'title': 'Full-cycle demo task (edited)', 'done': False}
3) MARK DONE - 200 {'id': 5, 'title': 'Full-cycle demo task (edited)', 'done': True}
4) DELETE - 204
5) GET /tasks - back to 3 tasks

Full CRUD lifecycle checkpoint completed - 201, 200, 200, 204, 200 in sequence.


## Task 5 (See it: Swagger UI)

This task involves building a simple RESTful Task API using FastAPI. The application manages an in-memory list of tasks and provides endpoints to create, retrieve, update, delete, filter, and reset tasks. It also includes health and statistics endpoints, demonstrating core FastAPI concepts, request handling, and basic API development practices.

In [20]:
app_py_source = '''\

# Task API - FlyRank W2 A1 (Python / FastAPI lane)
# Run with:  uvicorn app:app --reload --port 8000
# Docs at:   http://localhost:8000/docs


from typing import Optional
from fastapi import FastAPI, Response
from fastapi.responses import JSONResponse
from pydantic import BaseModel

app = FastAPI(title="Task API", version="1.0")

tasks = [
    {"id": 1, "title": "Buy milk", "done": False},
    {"id": 2, "title": "Write weekly report", "done": True},
    {"id": 3, "title": "Walk the dog", "done": False},
]
next_id = 4


def error(status_code: int, message: str) -> JSONResponse:
    return JSONResponse(status_code=status_code, content={"error": message})


def find_task(task_id: int):
    return next((t for t in tasks if t["id"] == task_id), None)


class TaskCreate(BaseModel):
    title: Optional[str] = None


class TaskUpdate(BaseModel):
    title: Optional[str] = None
    done: Optional[bool] = None


@app.get("/")
def root():
    return {"name": "Task API", "version": "1.0", "endpoints": ["/tasks"]}


@app.get("/health")
def health():
    return {"status": "ok"}


@app.get("/tasks")
def list_tasks(
    done: Optional[bool] = None,
    search: Optional[str] = None,
    limit: Optional[int] = None,
    offset: int = 0,
):
    result = tasks
    if done is not None:
        result = [t for t in result if t["done"] == done]
    if search:
        needle = search.lower()
        result = [t for t in result if needle in t["title"].lower()]
    if offset:
        result = result[offset:]
    if limit is not None:
        result = result[:limit]
    return result


@app.get("/tasks/{task_id}")
def get_task(task_id: int):
    task = find_task(task_id)
    if task is None:
        return error(404, f"Task {task_id} not found")
    return task


@app.post("/tasks", status_code=201)
def create_task(payload: TaskCreate):
    global next_id
    if not payload.title or not payload.title.strip():
        return error(400, "title is required and cannot be empty")
    new_task = {"id": next_id, "title": payload.title.strip(), "done": False}
    tasks.append(new_task)
    next_id += 1
    return JSONResponse(status_code=201, content=new_task)


@app.put("/tasks/{task_id}")
def update_task(task_id: int, payload: TaskUpdate):
    task = find_task(task_id)
    if task is None:
        return error(404, f"Task {task_id} not found")
    if payload.title is None and payload.done is None:
        return error(400, "request body must include at least 'title' or 'done'")
    if payload.title is not None and not payload.title.strip():
        return error(400, "title cannot be empty")
    if payload.title is not None:
        task["title"] = payload.title.strip()
    if payload.done is not None:
        task["done"] = payload.done
    return task


@app.delete("/tasks/{task_id}", status_code=204)
def delete_task(task_id: int):
    task = find_task(task_id)
    if task is None:
        return error(404, f"Task {task_id} not found")
    tasks.remove(task)
    return Response(status_code=204)


@app.get("/stats")
def stats():
    total = len(tasks)
    done_count = sum(1 for t in tasks if t["done"])
    return {"total": total, "done": done_count, "open": total - done_count}


@app.post("/reset")
def reset_tasks():
    global tasks, next_id
    tasks = [
        {"id": 1, "title": "Buy milk", "done": False},
        {"id": 2, "title": "Write weekly report", "done": True},
        {"id": 3, "title": "Walk the dog", "done": False},
    ]
    next_id = 4
    return {"message": "Tasks reset to the 3 example tasks", "tasks": tasks}


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open("app.py", "w", encoding="utf-8", newline="\n") as f:
    f.write(app_py_source)

print(f"Wrote app.py ({len(app_py_source.splitlines())} lines) as UTF-8.")
print("Run it with: uvicorn app:app --reload --port 8000")

Wrote app.py (134 lines) as UTF-8.
Run it with: uvicorn app:app --reload --port 8000


## extras (stretch goals)

In [21]:
clear_route(app, "/tasks", "GET")

@app.get("/tasks")
def list_tasks_extended(
    done: Optional[bool] = None,
    search: Optional[str] = None,
    limit: Optional[int] = None,
    offset: int = 0,
):
    result = tasks
    if done is not None:
        result = [t for t in result if t["done"] == done]
    if search:
        needle = search.lower()
        result = [t for t in result if needle in t["title"].lower()]
    if offset:
        result = result[offset:]
    if limit is not None:
        result = result[:limit]
    return result

@app.get("/stats")
def stats():
    total = len(tasks)
    done_count = sum(1 for t in tasks if t["done"])
    return {"total": total, "done": done_count, "open": total - done_count}

@app.post("/reset")
def reset_tasks():
    global tasks, next_id
    tasks = [
        {"id": 1, "title": "Buy milk", "done": False},
        {"id": 2, "title": "Write weekly report", "done": True},
        {"id": 3, "title": "Walk the dog", "done": False},
    ]
    next_id = 4
    return {"message": "Tasks reset to the 3 example tasks", "tasks": tasks}

client = TestClient(app)

print("GET /tasks?done=true -", client.get("/tasks", params={"done": "true"}).json())
print("GET /tasks?search=re -", client.get("/tasks", params={"search": "report"}).json())
print("GET /stats           -", client.get("/stats").json())
print("POST /reset          -", client.post("/reset").json())
print("GET /tasks?limit=1   -", client.get("/tasks", params={"limit": 1}).json())

print("\nStretch goals wired up and verified.")


GET /tasks?done=true - [{'id': 2, 'title': 'Write weekly report', 'done': True}]
GET /tasks?search=re - [{'id': 2, 'title': 'Write weekly report', 'done': True}]
GET /stats           - {'total': 3, 'done': 1, 'open': 2}
POST /reset          - {'message': 'Tasks reset to the 3 example tasks', 'tasks': [{'id': 1, 'title': 'Buy milk', 'done': False}, {'id': 2, 'title': 'Write weekly report', 'done': True}, {'id': 3, 'title': 'Walk the dog', 'done': False}]}
GET /tasks?limit=1   - [{'id': 1, 'title': 'Buy milk', 'done': False}]

Stretch goals wired up and verified.


In [15]:
# commiting checkpoint to git